# Research question answering functionality

This app demonstrated what happens in the different callbacks of `app/callbacks/rq_callbacks.py`.

In [1]:
%load_ext autoreload
%autoreload 2

This first chunk of code downloads data and saves it to a local directory called `mock_session_data/`.

In the first tab of the app, data gets uploaded and depending on the format, two outputs are saved:

* `raw_text_files.jsonl` if the input data was text (e.g. `.txt` or `.vtt`)
* `topic_modelling_data.csv`: this is data that has been converted to a tabular format for topic modelling, if it was not originally in a table.

Basically this first chunk mocks what would happen if you'd already gone through the upload step in the app.

In [ ]:
from discovery_qualfml import PROJECT_DIR
from discovery_qualfml.utils.llm_summarize import generate_summaries_and_quotes, create_final_output_df, create_output_excel
from discovery_qualfml.utils.llm_question_answering import format_transcripts, run_batch_check_for_all_rqs, create_batch_check_outputs_all_rqs

import boto3
import os

def download_s3_folder(bucket_name, s3_prefix, local_dir, aws_profile=None, region_name='eu-west-2'):
    """
    Downloads an S3 folder to a local directory.

    Args:
        bucket_name (str): Name of the S3 bucket.
        s3_prefix (str): Prefix of the "folder" in S3 to download.
        local_dir (str): Local path to save the downloaded files.
        aws_profile (str, optional): Named AWS CLI profile to use.
        region_name (str): AWS region.
    """
    if aws_profile:
        session = boto3.Session(profile_name=aws_profile, region_name=region_name)
    else:
        session = boto3.Session(region_name=region_name)

    s3 = session.resource('s3')
    bucket = s3.Bucket(bucket_name)

    for obj in bucket.objects.filter(Prefix=s3_prefix):
        rel_path = obj.key[len(s3_prefix):].lstrip('/')
        local_path = os.path.join(local_dir, rel_path)

        if obj.key.endswith('/'):  # Skip directories
            continue

        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        print(f'Downloading {obj.key} to {local_path}')
        bucket.download_file(obj.key, local_path)

# Example usage:
download_s3_folder(
    bucket_name='dsp-qualfml',
    s3_prefix='example_data/vtt_digestive/mock_session_data/',  # e.g., 'my-data/' (must end with / if it's a "folder")
    local_dir=f'{PROJECT_DIR}/outputs/mock_session_data/',
    aws_profile='default'  # Or None if using default credentials
)

/Users/rosie.oxbury/Documents/git_repos/discovery_qualfml/discovery_qualfml/utils/llm_summarize.py:20: LangChainDeprecationWarning: The class `AzureChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import AzureChatOpenAI``.
  llm = AzureChatOpenAI(


2025-06-18 15:41:51,247 - botocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials


The callbacks in the file `rq_callbacks.py` need to know the session id, so this is provided below as `"mock_session_data"`.

The first input in that tab is for research questions, which are inputted as a string with each RQ on a new line. We mock that input with the variable `rq_text` below.

In [3]:
session_id = "mock_session_data"

rq_text = """
How do participants feel about their diagnoses?
What are some difficulties participants have faced because of their conditions?
"""

The callbacks in `rq_callbacks.py` are as follows:
* `run_analysis()` does a bunch of things:
    - reformats the session data into the shape needed for the different LLM prompting steps
    - runs the different LLM prompting steps i.e. getting an answer to the RQ for each interview, creating a summary answer across all interviews, pulling out the best quotes from the interviews
    - formats the output into an excel that contains the summary answer to each RQ, the key quotes and the conversation/file identifiers for those quotes
* `download_excel()` makes the excel output available for download when a button is clicked
* `display_results()` displays the summary answers and key quotes
* `display_conversation()` brings up a popup with the full conversation for a given quote when clicked

The rest of this notebook focuses mainly on what's going on in the `run_analysis()` callback as that's where most of the heavy lifting happens.

The key input format for this tab is a Dict where the keys are conversation/file IDs, and the values are the full text of each interview/file, as you can see below:

In [4]:
conversation_text_dict = format_transcripts(session_id, 'source_file', text_col='transcript')
conversation_text_dict

text files


{'barbara.vtt': 'WEBVTT\nB: Bowel disorders, it is what it is.\nI: How old were you when you first started to realize you were having problems with your digestion?\nB: I was, uh, 21 and a half, to be exact, yeah.\nI: And do you suffer from celiac or, how would you define your discomfort?\nB: My colonoscopy says no celiac and not inflammatory bowel disease. My blood marker test says I have inflammatory bowel disease. I would label myself definitely gluten intolerant.\nI: And for the record, can you describe what that means? Gluten intolerance? As you would describe it.\nB: Gluten intolerance means that you, your body just does not digest or break down or really absorb gluten. And that is a protein that is found in wheat. I would also say that I don’t handle processed wheat well either. Um, and the symptoms are across the board. For me personally, um, I get, I’ll get joint pains, exhaustion, um, and I just feel incredibly full. After four or five bites, if I’m having, say, pasta or somet

At the first step, each conversation/file is presented to GPT and we ask for an answer to the RQ, and the key quotes that support that answer.

In [5]:
# Beware that this cell may show as complete before we've actually got back all the results from GPT
outdir = f"{PROJECT_DIR}/outputs/{session_id}"

output_paths, rq_dict = run_batch_check_for_all_rqs(
            rq_text=rq_text,
            conversation_text_dict = conversation_text_dict,
            output_dir=outdir,
        )

2025-06-18 15:41:51,850 - root - INFO - Using Azure OpenAI
2025-06-18 15:41:51,862 - langfuse - WARNING - Langfuse client is disabled since no public_key was provided as a parameter or environment variable 'LANGFUSE_PUBLIC_KEY'. See our docs: https://langfuse.com/docs/sdk/python/low-level-sdk#initialize-client
2025-06-18 15:41:51,874 - root - INFO - Using Azure OpenAI
2025-06-18 15:41:51,885 - langfuse - WARNING - Langfuse client is disabled since no public_key was provided as a parameter or environment variable 'LANGFUSE_PUBLIC_KEY'. See our docs: https://langfuse.com/docs/sdk/python/low-level-sdk#initialize-client


2025-06-18 15:41:51,899 - root - INFO - Processing batch 1/1
2025-06-18 15:41:51,901 - root - INFO - Processing batch 1/1


The next function just makes the results into a nice tabular format:

In [6]:
# If this cell errors, check that the `mock_session_data/` dir does indeed contain an output file for each RQ
batch_check_all_rqs = create_batch_check_outputs_all_rqs(conversation_text_dict, outdir, rq_dict)
batch_check_all_rqs

Quote NOT found in conversation text:
I’m not really sure. I don’t, I don’t know if they’re taught that in med school that, you know, that it’s not black and white.
Quote NOT found in conversation text:
it gives it a negative connotation, so I was never told celiac.
Accuracy of quotes found in original text: 85.71%
14
12
Quote NOT found in conversation text:
After four or five bites, I can’t eat anymore, feel nauseous.
Quote NOT found in conversation text:
It was really hard. With all the hormones and changing, It just kinda, I think, throws a lot of things off.
Quote NOT found in conversation text:
They wanted to send my sister to the psych unit. They thought it was a mental thing and we’re like, “No, she’s filling up the cup.”
Quote NOT found in conversation text:
It’s a disease on the inside that doesn’t always manifest on the outside.
Quote NOT found in conversation text:
I don’t like the fact I have to take this medication.
Quote NOT found in conversation text:
It’s so hard with t

,rq,id,answer,text,timestamp,model,temperature
0,How do participants feel about their diagnoses?,barbara.vtt,"Participants express a mix of frustration, acc...","Bowel disorders, it is what it is.",2025-06-18 14:41:51.901732+00:00,gpt-4o-mini,0
1,How do participants feel about their diagnoses?,barbara.vtt,"Participants express a mix of frustration, acc...",I just don’t like the fact I have to take this...,2025-06-18 14:41:51.901732+00:00,gpt-4o-mini,0
2,How do participants feel about their diagnoses?,barbara.vtt,"Participants express a mix of frustration, acc...","I think that’s what happened to me because of,...",2025-06-18 14:41:51.901732+00:00,gpt-4o-mini,0
3,How do participants feel about their diagnoses?,barbara.vtt,"Participants express a mix of frustration, acc...",I found that really hard for doctors. If it’s ...,2025-06-18 14:41:51.901732+00:00,gpt-4o-mini,0
4,How do participants feel about their diagnoses?,barbara.vtt,"Participants express a mix of frustration, acc...",They’re terrified.,2025-06-18 14:41:51.901732+00:00,gpt-4o-mini,0
5,How do participants feel about their diagnoses?,barbara.vtt,"Participants express a mix of frustration, acc...","I could get the flu really, really bad. And th...",2025-06-18 14:41:51.901732+00:00,gpt-4o-mini,0
6,How do participants feel about their diagnoses?,barbara.vtt,"Participants express a mix of frustration, acc...",I’m doing everything that I can in my power as...,2025-06-18 14:41:51.901732+00:00,gpt-4o-mini,0
7,How do participants feel about their diagnoses?,barbara.vtt,"Participants express a mix of frustration, acc...",You are your best advocate for yourself. Not t...,2025-06-18 14:41:51.901732+00:00,gpt-4o-mini,0
8,How do participants feel about their diagnoses?,sam.vtt,Participants express a complex relationship wi...,"once you put a label on it, you give it power.",2025-06-18 14:41:51.908500+00:00,gpt-4o-mini,0
9,How do participants feel about their diagnoses?,sam.vtt,Participants express a complex relationship wi...,I don’t like to use that label.,2025-06-18 14:41:51.908500+00:00,gpt-4o-mini,0


At the next step, the answers given per file and per RQ are summarised into a single answer per RQ, and from the quotes that were returned by `run_batch_check_for_all_rqs()`, the best/most illustrative ones are selected for each RQ.

In [7]:
summarize_and_quote_output = generate_summaries_and_quotes(rq_dict, batch_check_all_rqs)
summarize_and_quote_output

{'rq_1_20250618_154151': {'answer': 'Participants have a multifaceted perspective on their diagnoses, characterized by a blend of frustration, acceptance, and a sense of control. They often feel overwhelmed by the medical process and the stigma associated with their conditions, leading to hesitance in fully accepting diagnostic labels. However, they find comfort and empowerment in managing their health through lifestyle choices, such as diet and self-advocacy, which they view as integral to their emotional well-being. This complex relationship highlights both the challenges and the agency participants feel regarding their health.',
  'quotes': ['I just don’t like the fact I have to take this medication.',
   'They’re terrified.',
   'You are your best advocate for yourself. Not taking the medical. Being in tune. It’s clear if I’ve had a bad week. I eat foods I shouldn’t have been eating. I feel really lethargic and exhausted. If you don’t want to feel like this, don’t eat it. It’s pret

The final thing is to turn this output into a table format that can be converted to excel. The excel output also needs to contain conversation/file IDs for each quotes, so we check for exact matches between the quotes returned by `generate_summaries_and_quotes()` and the original text of each conversation/file.

In [8]:
summarise_output_df = create_final_output_df(rq_dict, conversation_text_dict, summarize_and_quote_output)

summarise_output_df.reset_index().to_csv(os.path.join(outdir, "summarise_output_df.csv"), index=False)

create_output_excel(summarise_output_df, outdir)

RQ ID: rq_1_20250618_154151
QUOTE:
I just don’t like the fact I have to take this medication.
→ FOUND IN: barbara.vtt

QUOTE:
They’re terrified.
→ FOUND IN: barbara.vtt

QUOTE:
You are your best advocate for yourself. Not taking the medical. Being in tune. It’s clear if I’ve had a bad week. I eat foods I shouldn’t have been eating. I feel really lethargic and exhausted. If you don’t want to feel like this, don’t eat it. It’s pretty immediate.
→ FOUND IN: barbara.vtt

QUOTE:
once you put a label on it, you give it power.
→ FOUND IN: sam.vtt

QUOTE:
I had to learn that that’s not my issue. Just because you want to drag me into your hell doesn’t mean that, and I had to learn that, how to be strong enough, to stand up for my own health, really.
→ FOUND IN: sam.vtt

RQ ID: rq_2_20250618_154151
QUOTE:
I get, I’ll get joint pains, exhaustion, um, and I just feel incredibly full.
→ FOUND IN: barbara.vtt

QUOTE:
I had, um, pretty bad hormonal acne, is what they would call it.
→ FOUND IN: barbar

This is what the tabular output looks like:

In [9]:
summarise_output_df

,question,answer,quote,source
0,How do participants feel about their diagnoses?,Participants have a multifaceted perspective o...,I just don’t like the fact I have to take this...,barbara.vtt
1,How do participants feel about their diagnoses?,Participants have a multifaceted perspective o...,They’re terrified.,barbara.vtt
2,How do participants feel about their diagnoses?,Participants have a multifaceted perspective o...,You are your best advocate for yourself. Not t...,barbara.vtt
3,How do participants feel about their diagnoses?,Participants have a multifaceted perspective o...,"once you put a label on it, you give it power.",sam.vtt
4,How do participants feel about their diagnoses?,Participants have a multifaceted perspective o...,I had to learn that that’s not my issue. Just ...,sam.vtt
5,What are some difficulties participants have f...,Participants have faced various difficulties d...,"I get, I’ll get joint pains, exhaustion, um, a...",barbara.vtt
6,What are some difficulties participants have f...,Participants have faced various difficulties d...,"I had, um, pretty bad hormonal acne, is what t...",barbara.vtt
7,What are some difficulties participants have f...,Participants have faced various difficulties d...,I’d just be exhausted. I’d have to take a nap....,sam.vtt
8,What are some difficulties participants have f...,Participants have faced various difficulties d...,"I haven’t lost friends, but I don’t hang out w...",sam.vtt
9,What are some difficulties participants have f...,Participants have faced various difficulties d...,"I remember the teachers would not, being, not ...",barbara.vtt
